# Data Preparation

This notebook takes the London smart meter readings from raw CSV files to a clean dataset ready for modelling.
We use **DuckDB** instead of pandas: the data has about 168 million rows, which is too big to load into memory comfortably. DuckDB runs SQL directly on files and only reads what each query needs.

## 1. Setup

Paths and the DuckDB connection live here so every later cell uses the same ones.

In [1]:
from pathlib import Path
import duckdb

# Find the repo root (the folder with pyproject.toml), so the paths work wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

RAW_CSVS = ROOT / "data/raw/halfhourly_dataset/*.csv"
RAW_PARQUET = ROOT / "data/interim/halfhourly_raw.parquet"
TYPED_PARQUET = ROOT / "data/interim/halfhourly_typed.parquet"
GRID_PARQUET = ROOT / "data/interim/halfhourly_grid.parquet"
CLEAN_PARQUET = ROOT / "data/interim/halfhourly_clean.parquet"
DAILY_PARQUET = ROOT / "data/processed/daily.parquet"
MONTHLY_PARQUET = ROOT / "data/processed/monthly.parquet"
MONTHLY_ENCODED_PARQUET = ROOT / "data/processed/monthly_encoded.parquet"

HOUSEHOLDS_CSV = ROOT / "data/raw/informations_households.csv"
WEATHER_CSV = ROOT / "data/raw/weather_hourly_darksky.csv"
HOLIDAYS_CSV = ROOT / "data/raw/uk_bank_holidays.csv"

# Cleaning thresholds (chosen in sections 7 and 8)
ZERO_RUN_MIN_SLOTS = 48      # a run of zeros this long (1 day) is treated as missing
MAX_INTERPOLATE_SLOTS = 4    # gaps up to this long (2 hours) are filled with a straight line
MIN_MONTH_COVERAGE = 0.90    # household-months with less real data than this are dropped
MAX_KWH_PER_SLOT = 11.5      # physical ceiling: a 100 A home supply (23 kW) running for 30 min
SPIKE_FACTOR = 10            # a reading this many times above its household's 99th percentile is a spike
STD_TARIFF_GBP_PER_KWH = 0.14228  # flat standard tariff, 14.228 p/kWh (London Datastore)

con = duckdb.connect()

## 2. Convert the CSVs to Parquet

The half-hourly readings come as 112 CSV files. Reading CSV is slow because every value is text and has to be parsed again on every query.

**Parquet** is a compressed file format that stores data by column and remembers each column's type. Converting once means every later query runs much faster and the file takes far less disk space.

This step only changes the format. We do **no cleaning** here: the values stay exactly as they are in the CSVs. The one change is renaming `energy(kWh/hh)` to `energy`, so it is easier to type in SQL.

The conversion takes a few minutes, so the cell skips it if the Parquet file already exists.

In [2]:
if RAW_PARQUET.exists():
    print("Parquet file already exists, skipping conversion.")
else:
    con.execute(f"""
        COPY (
            SELECT LCLid, tstp, "energy(kWh/hh)" AS energy
            FROM read_csv('{RAW_CSVS}')
        ) TO '{RAW_PARQUET}' (FORMAT parquet)
    """)
    print("Saved to", RAW_PARQUET.relative_to(ROOT))

Parquet file already exists, skipping conversion.


## 3. Check the conversion

We give the Parquet file a short name (`raw`) so later queries can say `FROM raw`.

The row count should match the integrity check: **167,817,133 lines − 112 header lines = 167,817,021 rows**. If it matches, no rows were lost in the conversion.

In [3]:
con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM '{RAW_PARQUET}'")

con.sql("SELECT count(*) AS rows FROM raw").show()
con.sql("SELECT * FROM raw LIMIT 5").show()

┌───────────┐
│   rows    │
│   int64   │
├───────────┤
│ 167817021 │
└───────────┘

┌───────────┬─────────────────────┬─────────┐
│   LCLid   │        tstp         │ energy  │
│  varchar  │      timestamp      │ varchar │
├───────────┼─────────────────────┼─────────┤
│ MAC000002 │ 2012-10-12 00:30:00 │  0      │
│ MAC000002 │ 2012-10-12 01:00:00 │  0      │
│ MAC000002 │ 2012-10-12 01:30:00 │  0      │
│ MAC000002 │ 2012-10-12 02:00:00 │  0      │
│ MAC000002 │ 2012-10-12 02:30:00 │  0      │
└───────────┴─────────────────────┴─────────┘



## 4. First look at the data

Before changing anything, we look at what the data actually contains. Everything found here decides what the cleaning steps need to do.

### 4.1 Column types and summary

`DESCRIBE` lists each column and its type (like pandas `.info()`). `SUMMARIZE` gives min, max, unique count, quartiles and the share of NULLs (like `.describe()`).

Watch the `energy` type. Energy readings should be numbers, so if DuckDB stores them as text (`VARCHAR`), some values in that column are not numbers.

In [4]:
con.sql("DESCRIBE raw").show()

# .df() shows SUMMARIZE as a full table instead of cutting off columns
con.sql("SUMMARIZE raw").df()

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ LCLid       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tstp        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ energy      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,LCLid,VARCHAR,MAC000002,MAC005567,7122,NaN,<NA>,NaN,NaN,NaN,167817021,0.0
1,tstp,TIMESTAMP,2011-11-23 09:00:00,2014-02-28 00:00:00,38313,2013-03-28 05:28:20.296595,<NA>,2012-10-21 20:23:34.721176,2013-03-30 06:43:46.879906,2013-09-10 08:09:56.366297,167817021,0.0
2,energy,VARCHAR,0,Null,6458,NaN,<NA>,NaN,NaN,NaN,167817021,0.0


### 4.2 What kinds of values each column holds

`SUMMARIZE` only counts real NULLs. A missing reading written as the text `Null` counts as a normal value, so it can hide there. Here we sort every value into a category and count each one.

**Energy:** the values have spaces around them (like `' 0 '`), so we `trim` them first. `TRY_CAST` turns a value into a number, or gives NULL if it can't, which lets us spot the values that are not numbers.

In [5]:
con.sql("""
    SELECT
        CASE
            WHEN energy IS NULL                            THEN 'NULL'
            WHEN trim(energy) = ''                         THEN 'empty'
            WHEN TRY_CAST(trim(energy) AS DOUBLE) IS NULL  THEN 'not a number'
            WHEN TRY_CAST(trim(energy) AS DOUBLE) < 0      THEN 'negative'
            WHEN TRY_CAST(trim(energy) AS DOUBLE) = 0      THEN 'zero'
            ELSE 'positive'
        END AS value_type,
        count(*) AS rows,
        any_value(energy) AS example
    FROM raw
    GROUP BY value_type
    ORDER BY rows DESC
""").show()

# Every distinct value that is not a number, so nothing hides behind the one example above
con.sql("""
    SELECT energy AS value, count(*) AS rows
    FROM raw
    WHERE TRY_CAST(trim(energy) AS DOUBLE) IS NULL
    GROUP BY energy
""").show()

┌──────────────┬───────────┬─────────────┐
│  value_type  │   rows    │   example   │
│   varchar    │   int64   │   varchar   │
├──────────────┼───────────┼─────────────┤
│ positive     │ 165809909 │  2.2309999  │
│ zero         │   2001552 │  0          │
│ not a number │      5560 │ Null        │
└──────────────┴───────────┴─────────────┘

┌─────────┬───────┐
│  value  │ rows  │
│ varchar │ int64 │
├─────────┼───────┤
│ Null    │  5560 │
└─────────┴───────┘



**Household ID (`LCLid`):** every ID should be `MAC` followed by 6 digits.

In [6]:
con.sql("""
    SELECT
        CASE
            WHEN LCLid IS NULL                          THEN 'NULL'
            WHEN regexp_full_match(LCLid, 'MAC[0-9]{6}') THEN 'MAC + 6 digits'
            ELSE 'other format'
        END AS value_type,
        count(*) AS rows,
        any_value(LCLid) AS example
    FROM raw
    GROUP BY value_type
""").show()

┌────────────────┬───────────┬───────────┐
│   value_type   │   rows    │  example  │
│    varchar     │   int64   │  varchar  │
├────────────────┼───────────┼───────────┤
│ MAC + 6 digits │ 167817021 │ MAC003817 │
└────────────────┴───────────┴───────────┘



**Timestamp (`tstp`):** meters report every 30 minutes, so each timestamp should land exactly on `:00` or `:30`. `time_bucket` rounds a timestamp down to the half hour. If rounding changes it, the reading is off the half hour.

In [7]:
con.sql("""
    SELECT
        CASE
            WHEN tstp IS NULL                                  THEN 'NULL'
            WHEN tstp = time_bucket(INTERVAL 30 MINUTE, tstp)  THEN 'on the half hour'
            ELSE 'off the half hour'
        END AS value_type,
        count(*) AS rows,
        any_value(tstp) AS example
    FROM raw
    GROUP BY value_type
""").show()

┌───────────────────┬───────────┬─────────────────────┐
│    value_type     │   rows    │       example       │
│      varchar      │   int64   │      timestamp      │
├───────────────────┼───────────┼─────────────────────┤
│ on the half hour  │ 167811461 │ 2013-02-04 11:00:00 │
│ off the half hour │      5560 │ 2012-12-18 15:24:03 │
└───────────────────┴───────────┴─────────────────────┘



### 4.3 Households and how long each was recorded

Households joined and left the trial at different times. We find each household's first and last reading, then summarise the spread. This matters for forecasting: a household with only a few months of data gives the model little history to learn from.

In [8]:
con.sql("""
    WITH per_household AS (
        SELECT LCLid, min(tstp) AS first_reading, max(tstp) AS last_reading
        FROM raw
        GROUP BY LCLid
    )
    SELECT
        count(*)                AS households,
        min(first_reading)      AS earliest_start,
        max(first_reading)      AS latest_start,
        min(last_reading)       AS earliest_end,
        max(last_reading)       AS latest_end,
        count(*) FILTER (WHERE date_diff('day', first_reading, last_reading) < 365) AS under_one_year
    FROM per_household
""").show()

┌────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────────────┐
│ households │   earliest_start    │    latest_start     │    earliest_end     │     latest_end      │ under_one_year │
│   int64    │      timestamp      │      timestamp      │      timestamp      │      timestamp      │     int64      │
├────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼────────────────┤
│       5566 │ 2011-11-23 09:00:00 │ 2013-10-29 18:23:02 │ 2012-11-06 23:30:00 │ 2014-02-28 00:00:00 │            184 │
└────────────┴─────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴────────────────┘



## 5. Fix data types and formatting

Section 4 found two problems: 5,560 energy values are the text `Null`, and 5,560 timestamps fall off the half hour. Here we check how they relate, confirm how the timestamps handle daylight saving, and then save a version of the data with the right types.

### 5.1 Are the `Null` values and the off-grid timestamps the same rows?

If every `Null` row is also off the half hour, it is one problem, not two. We also look at which days these rows fall on.

In [9]:
con.sql("""
    SELECT
        count(*) FILTER (WHERE TRY_CAST(trim(energy) AS DOUBLE) IS NULL)         AS not_a_number,
        count(*) FILTER (WHERE tstp <> time_bucket(INTERVAL 30 MINUTE, tstp))    AS off_half_hour,
        count(*) FILTER (WHERE TRY_CAST(trim(energy) AS DOUBLE) IS NULL
                           AND tstp <> time_bucket(INTERVAL 30 MINUTE, tstp))    AS both_problems
    FROM raw
""").show()

# Which days the bad rows fall on
con.sql("""
    SELECT tstp::DATE AS day, count(*) AS rows
    FROM raw
    WHERE TRY_CAST(trim(energy) AS DOUBLE) IS NULL
    GROUP BY day
    ORDER BY rows DESC
""").show()

┌──────────────┬───────────────┬───────────────┐
│ not_a_number │ off_half_hour │ both_problems │
│    int64     │     int64     │     int64     │
├──────────────┼───────────────┼───────────────┤
│         5560 │          5560 │          5560 │
└──────────────┴───────────────┴───────────────┘

┌────────────┬───────┐
│    day     │ rows  │
│    date    │ int64 │
├────────────┼───────┤
│ 2012-12-18 │  5521 │
│ 2012-12-19 │    26 │
│ 2013-07-05 │     3 │
│ 2013-10-29 │     3 │
│ 2012-12-20 │     1 │
│ 2013-04-16 │     1 │
│ 2013-01-04 │     1 │
│ 2013-02-06 │     1 │
│ 2013-03-08 │     1 │
│ 2013-02-13 │     1 │
│ 2013-09-20 │     1 │
└────────────┴───────┘
  11 rows  2 columns



All 5,560 bad rows have both problems, and almost all of them land on one day (2012-12-18). That pattern points to a logging glitch, not real meter readings. The rows carry no energy value, so there is nothing to recover. **We decide to drop them.**

### 5.2 Do the timestamps follow daylight saving?

The UK moves its clocks twice a year. If timestamps were in local time, the spring day would have 46 half hours and the autumn day 50. If they are in UTC/GMT, every day has 48. We count readings per household on each clock-change day.

In [10]:
# UK clock-change days within the data's date range
CLOCK_CHANGE_DAYS = ['2012-03-25', '2012-10-28', '2013-03-31', '2013-10-27']

con.sql(f"""
    WITH per_household AS (
        SELECT LCLid, tstp::DATE AS day, count(*) AS readings
        FROM raw
        WHERE tstp::DATE IN {tuple(CLOCK_CHANGE_DAYS)}
        GROUP BY LCLid, day
    )
    SELECT
        day,
        count(*) FILTER (WHERE readings = 46)  AS households_with_46,
        count(*) FILTER (WHERE readings = 48)  AS households_with_48,
        count(*) FILTER (WHERE readings = 50)  AS households_with_50,
        count(*) FILTER (WHERE readings NOT IN (46, 48, 50)) AS other
    FROM per_household
    GROUP BY day
    ORDER BY day
""").show()

┌────────────┬────────────────────┬────────────────────┬────────────────────┬───────┐
│    day     │ households_with_46 │ households_with_48 │ households_with_50 │ other │
│    date    │       int64        │       int64        │       int64        │ int64 │
├────────────┼────────────────────┼────────────────────┼────────────────────┼───────┤
│ 2012-03-25 │                  0 │               1185 │                  0 │     4 │
│ 2012-10-28 │                  0 │               5502 │                  0 │    18 │
│ 2013-03-31 │                  0 │               5399 │                  0 │    11 │
│ 2013-10-27 │                  1 │               5176 │                  0 │    17 │
└────────────┴────────────────────┴────────────────────┴────────────────────┴───────┘



Almost every household has exactly 48 readings on every clock-change day, so the timestamps are in **UTC/GMT** and every day has 48 half hours. This keeps the later steps simple: no day needs special handling. The few `other` households are ones with gaps that day, which section 6 deals with.

### 5.3 Save the data with the right types


We drop the `Null` rows, then convert `energy` to a number. Once the `Null` rows are gone every value is numeric, so we use a plain `CAST`: if anything unexpected were left, it would fail loudly instead of silently becoming NULL. We rename the column to `kwh` because each value is the energy used in that half hour, in kWh.

In [11]:
con.execute(f"""
    COPY (
        SELECT LCLid, tstp, CAST(trim(energy) AS DOUBLE) AS kwh
        FROM raw
        WHERE TRY_CAST(trim(energy) AS DOUBLE) IS NOT NULL
    ) TO '{TYPED_PARQUET}' (FORMAT parquet)
""")

con.execute(f"CREATE OR REPLACE VIEW typed AS SELECT * FROM '{TYPED_PARQUET}'")

### 5.4 Check the result

We compare row and household counts before and after, list the households that were removed, and confirm the new types.

In [12]:
con.sql("""
    SELECT
        (SELECT count(*) FROM raw)               AS rows_before,
        (SELECT count(*) FROM typed)             AS rows_after,
        (SELECT count(DISTINCT LCLid) FROM raw)  AS households_before,
        (SELECT count(DISTINCT LCLid) FROM typed) AS households_after
""").show()

# The households that had no usable readings at all
con.sql("SELECT DISTINCT LCLid FROM raw EXCEPT SELECT DISTINCT LCLid FROM typed ORDER BY LCLid").show()

con.sql("DESCRIBE typed").show()

┌─────────────┬────────────┬───────────────────┬──────────────────┐
│ rows_before │ rows_after │ households_before │ households_after │
│    int64    │   int64    │       int64       │      int64       │
├─────────────┼────────────┼───────────────────┼──────────────────┤
│   167817021 │  167811461 │              5566 │             5561 │
└─────────────┴────────────┴───────────────────┴──────────────────┘

┌───────────┐
│   LCLid   │
│  varchar  │
├───────────┤
│ MAC001150 │
│ MAC005556 │
│ MAC005559 │
│ MAC005560 │
│ MAC005563 │
└───────────┘

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ LCLid       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tstp        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ kwh         │ DOUBLE      │ 

## 6. Duplicates

A household should have at most one reading per half hour. We group by household and timestamp and count the pairs that appear more than once. Checking the pair, not the whole row, also catches repeats where the energy values disagree.

In [13]:
con.sql("""
    SELECT count(*) AS repeated_keys
    FROM (
        SELECT LCLid, tstp
        FROM typed
        GROUP BY LCLid, tstp
        HAVING count(*) > 1
    )
""").show()

┌───────────────┐
│ repeated_keys │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘



No household has two readings for the same half hour, so there is nothing to remove.

## 7. Missing data: explore

A missing reading doesn't show up as an empty value. The row is simply not there. To see the gaps, we first build the full list of half hours each household *should* have, then check which ones have a reading.

### 7.1 Build the full half-hour grid

For each household we take its first and last reading and list every half hour in between (`generate_series`). Then we join the readings onto that list. A half hour with no reading ends up with `kwh` = NULL.

This creates about 168 million rows, so we save it to Parquet and read it back from there.

In [14]:
con.execute(f"""
    COPY (
        WITH ranges AS (
            SELECT LCLid, min(tstp) AS first_reading, max(tstp) AS last_reading
            FROM typed
            GROUP BY LCLid
        ),
        slots AS (
            SELECT LCLid, unnest(generate_series(first_reading, last_reading, INTERVAL 30 MINUTE)) AS tstp
            FROM ranges
        )
        SELECT slots.LCLid, slots.tstp, typed.kwh
        FROM slots
        LEFT JOIN typed USING (LCLid, tstp)
    ) TO '{GRID_PARQUET}' (FORMAT parquet)
""")

con.execute(f"CREATE OR REPLACE VIEW grid AS SELECT * FROM '{GRID_PARQUET}'")

### 7.2 How much is missing

`count(*)` counts every slot and `count(kwh)` counts only slots with a reading, so the difference is the number of missing slots.

In [15]:
con.sql("""
    SELECT
        count(*)                                              AS expected_slots,
        count(kwh)                                            AS readings,
        count(*) - count(kwh)                                 AS missing_slots,
        round(100.0 * (count(*) - count(kwh)) / count(*), 3)  AS missing_pct
    FROM grid
""").show()

con.sql("""
    WITH per_household AS (
        SELECT LCLid, count(*) - count(kwh) AS missing_slots
        FROM grid
        GROUP BY LCLid
    )
    SELECT
        count(*)                                  AS households,
        count(*) FILTER (WHERE missing_slots > 0) AS households_with_gaps
    FROM per_household
""").show()

┌────────────────┬───────────┬───────────────┬─────────────┐
│ expected_slots │ readings  │ missing_slots │ missing_pct │
│     int64      │   int64   │     int64     │   double    │
├────────────────┼───────────┼───────────────┼─────────────┤
│      168268119 │ 167811461 │        456658 │       0.271 │
└────────────────┴───────────┴───────────────┴─────────────┘

┌────────────┬──────────────────────┐
│ households │ households_with_gaps │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│       5561 │                 5300 │
└────────────┴──────────────────────┘



### 7.3 How long the gaps are

The share of missing slots doesn't tell us how to fix them. A single missing half hour is easy to fill, but a missing month is not. So we group consecutive missing slots into **runs** and count runs by length.

The two helper functions below do this. `find_runs` works for any condition, so we reuse it for zero runs in 7.4.

In [16]:
def find_runs(name, source, condition):
    """Save a view `name` with one row per run of consecutive half hours in `source` where `condition` is true."""
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT LCLid, min(tstp) AS run_start, count(*) AS slots
        FROM (
            SELECT
                LCLid,
                tstp,
                -- consecutive slots step up 30 min at a time, and so does row_number,
                -- so this value stays the same for every slot in one run
                tstp - INTERVAL 30 MINUTE * row_number() OVER (PARTITION BY LCLid ORDER BY tstp) AS run_id
            FROM {source}
            WHERE {condition}
        )
        GROUP BY LCLid, run_id
    """)


def run_lengths(name):
    """Group the runs in view `name` by length."""
    return con.sql(f"""
        SELECT
            CASE
                WHEN slots = 1     THEN '1. 30 min'
                WHEN slots <= 4    THEN '2. up to 2 hours'
                WHEN slots < 48    THEN '3. under 1 day'
                WHEN slots < 1440  THEN '4. 1 to 30 days'
                ELSE                    '5. 30 days or more'
            END AS run_length,
            count(*)              AS runs,
            sum(slots)            AS slots,
            count(DISTINCT LCLid) AS households
        FROM {name}
        GROUP BY run_length
        ORDER BY run_length
    """)

In [17]:
find_runs("gaps", "grid", "kwh IS NULL")

run_lengths("gaps").show()
con.sql("SELECT max(slots) AS longest_gap_slots, round(max(slots) / 48, 1) AS longest_gap_days FROM gaps").show()

┌────────────────────┬───────┬────────┬────────────┐
│     run_length     │ runs  │ slots  │ households │
│      varchar       │ int64 │ int128 │   int64    │
├────────────────────┼───────┼────────┼────────────┤
│ 1. 30 min          │ 16870 │  16870 │       4980 │
│ 2. up to 2 hours   │  1274 │   3346 │        994 │
│ 3. under 1 day     │  1395 │  13282 │        935 │
│ 4. 1 to 30 days    │  5859 │ 386008 │       2680 │
│ 5. 30 days or more │     7 │  37152 │          7 │
└────────────────────┴───────┴────────┴────────────┘

┌───────────────────┬──────────────────┐
│ longest_gap_slots │ longest_gap_days │
│       int64       │      double      │
├───────────────────┼──────────────────┤
│             10080 │            210.0 │
└───────────────────┴──────────────────┘



### 7.4 Long runs of zeros

A reading of 0 kWh is valid for one half hour. But a home almost never uses exactly zero for a whole day, because a fridge alone draws power. Long runs of zeros usually mean a faulty meter or a home that was empty, so they behave more like missing data than real usage. We count them the same way as the gaps.

In [18]:
find_runs("zero_runs", "grid", "kwh = 0")

run_lengths("zero_runs").show()
con.sql("""
    SELECT
        count(DISTINCT LCLid) FILTER (WHERE slots >= 48)   AS households_with_1_day_or_more,
        count(DISTINCT LCLid) FILTER (WHERE slots >= 1440) AS households_with_30_days_or_more
    FROM zero_runs
""").show()

┌────────────────────┬────────┬────────┬────────────┐
│     run_length     │  runs  │ slots  │ households │
│      varchar       │ int64  │ int128 │   int64    │
├────────────────────┼────────┼────────┼────────────┤
│ 1. 30 min          │ 379207 │ 379207 │       1266 │
│ 2. up to 2 hours   │ 154668 │ 387073 │       1007 │
│ 3. under 1 day     │  37303 │ 431636 │        649 │
│ 4. 1 to 30 days    │   1870 │ 462738 │        226 │
│ 5. 30 days or more │    116 │ 340898 │         52 │
└────────────────────┴────────┴────────┴────────────┘

┌───────────────────────────────┬─────────────────────────────────┐
│ households_with_1_day_or_more │ households_with_30_days_or_more │
│             int64             │              int64              │
├───────────────────────────────┼─────────────────────────────────┤
│                           231 │                              52 │
└───────────────────────────────┴─────────────────────────────────┘



### 7.5 When do the long gaps happen?

Most missing slots sit in gaps of a day or more. If those were random meter faults, they would be spread across many different days. Here we check which days the long gaps start on, and how long they usually are.

In [19]:
# Days on which the most households start a gap of a day or more
con.sql("""
    SELECT run_start::DATE AS day, count(*) AS households
    FROM gaps
    WHERE slots >= 48
    GROUP BY day
    ORDER BY households DESC
    LIMIT 10
""").show()

# The most common lengths of those long gaps
con.sql("""
    SELECT slots, slots / 48 AS days, count(*) AS gaps
    FROM gaps
    WHERE slots >= 48
    GROUP BY slots
    ORDER BY gaps DESC
    LIMIT 5
""").show()

┌────────────┬────────────┐
│    day     │ households │
│    date    │   int64    │
├────────────┼────────────┤
│ 2012-06-13 │        737 │
│ 2012-06-15 │        632 │
│ 2014-01-19 │        569 │
│ 2012-12-11 │        441 │
│ 2012-06-16 │        438 │
│ 2012-05-16 │        296 │
│ 2012-06-14 │        167 │
│ 2012-07-03 │         61 │
│ 2012-11-08 │         21 │
│ 2014-01-12 │         15 │
└────────────┴────────────┘
  10 rows       2 columns

┌───────┬────────┬───────┐
│ slots │  days  │ gaps  │
│ int64 │ double │ int64 │
├───────┼────────┼───────┤
│    48 │    1.0 │  4641 │
│    96 │    2.0 │   892 │
│   144 │    3.0 │   172 │
│   192 │    4.0 │    70 │
│   240 │    5.0 │    16 │
└───────┴────────┴───────┘



The long gaps cluster on a handful of dates, with hundreds of households missing the same day, and most last exactly 1 or 2 days. That points to the trial's data collection going down for whole days, not to individual meters failing. It also shapes the fix: a straight line across a whole missing day would erase the daily usage pattern, so these gaps need a different fill.

## 8. Missing data: decide and apply

Based on section 7, we handle missing data in four steps. The thresholds are in the setup cell.

1. **Treat long zero runs as missing.** A run of zeros lasting a day or more points to a meter fault, so we set those readings to NULL. Shorter zero runs stay, since they can be real low usage.
2. **Keep only months with enough real data.** Our target is a monthly bill, so we judge quality per household per month. A month is kept if it is a full calendar month for that household and at least 90% of its half hours have a real reading.
3. **Fill short gaps (up to 2 hours) with a straight line** between the readings on either side. Usage changes little over 2 hours.
4. **Fill the remaining gaps with the household's usual usage for that time of day,** using its average for the same half hour in the same month. This keeps the daily pattern, which matters for the 1-day outages.

### 8.1 Treat long zero runs as missing

In [20]:
# The long zero runs, saved as a small table of time ranges
con.execute(f"""
    CREATE OR REPLACE TABLE long_zero_runs AS
    SELECT LCLid, run_start, run_start + INTERVAL 30 MINUTE * (slots - 1) AS run_end
    FROM zero_runs
    WHERE slots >= {ZERO_RUN_MIN_SLOTS}
""")

# Same as the grid, but slots inside a long zero run get kwh = NULL
con.execute("""
    CREATE OR REPLACE VIEW masked AS
    SELECT
        grid.LCLid,
        grid.tstp,
        CASE WHEN z.LCLid IS NULL THEN grid.kwh END AS kwh
    FROM grid
    LEFT JOIN long_zero_runs AS z
        ON grid.LCLid = z.LCLid
       AND grid.tstp BETWEEN z.run_start AND z.run_end
""")

con.sql("""
    SELECT count(*) AS slots, count(kwh) AS real_readings, count(*) - count(kwh) AS missing_after_masking
    FROM masked
""").show()

┌───────────┬───────────────┬───────────────────────┐
│   slots   │ real_readings │ missing_after_masking │
│   int64   │     int64     │         int64         │
├───────────┼───────────────┼───────────────────────┤
│ 168268119 │     167007825 │               1260294 │
└───────────┴───────────────┴───────────────────────┘



### 8.2 Choose the monthly coverage cutoff

For each household-month we compare real readings with the number of half hours in that calendar month. Months where the household started or stopped partway through never count, because their bill would be incomplete. We compare a few cutoffs to see how much data each one keeps.

In [21]:
con.execute("""
    CREATE OR REPLACE TABLE month_coverage AS
    SELECT
        LCLid,
        date_trunc('month', tstp)            AS month,
        48 * day(last_day(min(tstp)))        AS slots_in_month,
        count(*)                             AS slots_in_grid,
        count(kwh)                           AS real_readings,
        count(kwh) / (48 * day(last_day(min(tstp))))  AS coverage
    FROM masked
    GROUP BY LCLid, month
""")

# How many household-months each cutoff keeps (partial first/last months never count)
con.sql("""
    SELECT
        cutoff,
        count(*) FILTER (WHERE coverage >= cutoff AND slots_in_grid = slots_in_month)               AS months_kept,
        count(*)                                                                                     AS months_total,
        round(100.0 * months_kept / months_total, 1)                                                 AS pct_kept,
        count(DISTINCT LCLid) FILTER (WHERE coverage >= cutoff AND slots_in_grid = slots_in_month)  AS households_kept
    FROM month_coverage, (VALUES (0.80), (0.90), (0.95), (0.99)) AS t(cutoff)
    GROUP BY cutoff
    ORDER BY cutoff
""").show()

┌──────────────┬─────────────┬──────────────┬──────────┬─────────────────┐
│    cutoff    │ months_kept │ months_total │ pct_kept │ households_kept │
│ decimal(3,2) │    int64    │    int64     │  double  │      int64      │
├──────────────┼─────────────┼──────────────┼──────────┼─────────────────┤
│         0.80 │      106510 │       118482 │     89.9 │            5548 │
│         0.90 │      106225 │       118482 │     89.7 │            5548 │
│         0.95 │      105194 │       118482 │     88.8 │            5548 │
│         0.99 │      102862 │       118482 │     86.8 │            5548 │
└──────────────┴─────────────┴──────────────┴──────────┴─────────────────┘



The cutoff barely changes the result: going from 80% to 99% drops only about 3,600 of 118,482 household-months. Most of the roughly 12,000 months removed are partial first or last months, which no cutoff can keep. We use **90%**: at most about 3 days of a month's bill is estimated, and it keeps almost every complete month.

### 8.3 Fill short gaps with a straight line

We find the gaps again (now including the masked zero runs). For each gap of up to 2 hours, we take the reading just before and just after it and space the missing values evenly between them. For example, a single missing slot between 0.2 and 0.4 kWh becomes 0.3 kWh.

In [22]:
find_runs("masked_gaps", "masked", "kwh IS NULL")

con.execute(f"""
    CREATE OR REPLACE TABLE interpolated AS
    WITH short_gaps AS (
        -- each short gap, with the reading just before it and just after it
        SELECT
            g.LCLid,
            g.run_start,
            g.slots,
            before.kwh AS kwh_before,
            after.kwh  AS kwh_after
        FROM masked_gaps AS g
        JOIN masked AS before ON before.LCLid = g.LCLid AND before.tstp = g.run_start - INTERVAL 30 MINUTE
        JOIN masked AS after  ON after.LCLid  = g.LCLid AND after.tstp  = g.run_start + INTERVAL 30 MINUTE * g.slots
        WHERE g.slots <= {MAX_INTERPOLATE_SLOTS}
    ),
    steps AS (
        -- one row per missing slot: step 1, 2, ... up to the gap length
        SELECT *, unnest(generate_series(1, slots)) AS step
        FROM short_gaps
    )
    SELECT
        LCLid,
        run_start + INTERVAL 30 MINUTE * (step - 1)                     AS tstp,
        kwh_before + (kwh_after - kwh_before) * step / (slots + 1)      AS kwh
    FROM steps
""")

con.sql("SELECT count(*) AS interpolated_slots FROM interpolated").show()

┌────────────────────┐
│ interpolated_slots │
│       int64        │
├────────────────────┤
│              20126 │
└────────────────────┘



### 8.4 Fill the remaining gaps and save the clean data

We keep only the months that passed the coverage cutoff. Each slot then takes the first value available: the real reading, the straight-line value, or the household's average for that half hour in that month. A `status` column records which one it is, so we always know which values are real.

In [23]:
con.execute(f"""
    COPY (
        WITH kept_months AS (
            SELECT LCLid, month
            FROM month_coverage
            WHERE coverage >= {MIN_MONTH_COVERAGE} AND slots_in_grid = slots_in_month
        ),
        slots AS (
            SELECT
                m.LCLid,
                m.tstp,
                k.month,
                m.kwh AS real_kwh,
                i.kwh AS interpolated_kwh
            FROM masked AS m
            JOIN kept_months AS k   ON k.LCLid = m.LCLid AND k.month = date_trunc('month', m.tstp)
            LEFT JOIN interpolated AS i ON i.LCLid = m.LCLid AND i.tstp = m.tstp
        ),
        profile AS (
            -- each household's average reading for each half hour of the day, per month (real readings only)
            SELECT LCLid, month, CAST(tstp AS TIME) AS time_of_day, avg(real_kwh) AS avg_kwh
            FROM slots
            GROUP BY LCLid, month, time_of_day
        )
        SELECT
            s.LCLid,
            s.tstp,
            coalesce(s.real_kwh, s.interpolated_kwh, p.avg_kwh) AS kwh,
            CASE
                WHEN s.real_kwh IS NOT NULL         THEN 'real'
                WHEN s.interpolated_kwh IS NOT NULL THEN 'interpolated'
                ELSE 'profile_filled'
            END AS status
        FROM slots AS s
        JOIN profile AS p
          ON p.LCLid = s.LCLid AND p.month = s.month AND p.time_of_day = CAST(s.tstp AS TIME)
    ) TO '{CLEAN_PARQUET}' (FORMAT parquet)
""")

con.execute(f"CREATE OR REPLACE VIEW clean AS SELECT * FROM '{CLEAN_PARQUET}'")

### 8.5 Check the result

Every slot should now have a value, and almost all of them should be real readings.

In [24]:
con.sql("""
    SELECT
        status,
        count(*)                                           AS slots,
        round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS pct
    FROM clean
    GROUP BY status
    ORDER BY slots DESC
""").show()

con.sql("""
    SELECT
        count(*)                                           AS rows,
        count(*) - count(kwh)                              AS null_kwh,
        count(DISTINCT LCLid)                              AS households,
        count(DISTINCT (LCLid, date_trunc('month', tstp))) AS household_months
    FROM clean
""").show()

┌────────────────┬───────────┬────────┐
│     status     │   slots   │  pct   │
│    varchar     │   int64   │ double │
├────────────────┼───────────┼────────┤
│ real           │ 155267623 │ 99.844 │
│ profile_filled │    224920 │  0.145 │
│ interpolated   │     18097 │  0.012 │
└────────────────┴───────────┴────────┘

┌───────────┬──────────┬────────────┬──────────────────┐
│   rows    │ null_kwh │ households │ household_months │
│   int64   │  int64   │   int64    │      int64       │
├───────────┼──────────┼────────────┼──────────────────┤
│ 155510640 │        0 │       5548 │           106225 │
└───────────┴──────────┴────────────┴──────────────────┘



Every slot now has a value, and 99.84% of them are real readings. The clean data has 155,510,640 half hours across 5,548 households and 106,225 household-months. 13 households drop out completely because none of their months passed the coverage cutoff.

## 9. Outliers

An outlier is a reading far away from the rest. Some are errors, but in energy data many are real: an electric shower, a heater or an oven can push one half hour well above normal. We check two things and then decide whether to keep, fix or remove them.

### 9.1 Values that can't be real

Energy use can't be negative. It also can't go above what the home's supply can deliver: a typical UK home supply is 100 A at 230 V, about 23 kW, which is at most 11.5 kWh in half an hour. Anything outside these limits would be a meter error.

In [25]:
con.sql(f"""
    SELECT
        count(*) FILTER (WHERE kwh < 0)                   AS negative,
        count(*) FILTER (WHERE kwh > {MAX_KWH_PER_SLOT})  AS above_ceiling,
        max(kwh)                                          AS highest_reading
    FROM clean
""").show()

┌──────────┬───────────────┬─────────────────┐
│ negative │ above_ceiling │ highest_reading │
│  int64   │     int64     │     double      │
├──────────┼───────────────┼─────────────────┤
│        0 │             0 │      10.7609997 │
└──────────┴───────────────┴─────────────────┘



No reading is negative or above the ceiling. The highest reading is about 10.8 kWh in half an hour, which is high but possible.

### 9.2 Spikes compared with the household's own usage

A reading that is normal for a large home can be extreme for a small flat, so we compare each reading with **its own household's** usage. For each household we take the 99th percentile of its real readings (99% of its readings are at or below this value). A reading more than 10 times that value counts as a spike.

In [26]:
# Each household's 99th percentile: 99% of its real readings are at or below this value
con.execute("""
    CREATE OR REPLACE TABLE household_p99 AS
    SELECT LCLid, quantile_cont(kwh, 0.99) AS p99
    FROM clean
    WHERE status = 'real'
    GROUP BY LCLid
""")

con.execute(f"""
    CREATE OR REPLACE TABLE spikes AS
    SELECT c.LCLid, c.tstp, c.kwh, p.p99, round(c.kwh / p.p99, 1) AS times_p99
    FROM clean AS c
    JOIN household_p99 AS p USING (LCLid)
    WHERE p.p99 > 0 AND c.kwh > {SPIKE_FACTOR} * p.p99
""")

con.sql("SELECT count(*) AS spikes, count(DISTINCT LCLid) AS households FROM spikes").show()
con.sql("SELECT * FROM spikes ORDER BY times_p99 DESC LIMIT 10").show()

┌────────┬────────────┐
│ spikes │ households │
│ int64  │   int64    │
├────────┼────────────┤
│     34 │         14 │
└────────┴────────────┘

┌───────────┬─────────────────────┬───────────┬─────────────────────┬───────────┐
│   LCLid   │        tstp         │    kwh    │         p99         │ times_p99 │
│  varchar  │      timestamp      │  double   │       double        │  double   │
├───────────┼─────────────────────┼───────────┼─────────────────────┼───────────┤
│ MAC001349 │ 2012-08-06 14:30:00 │     1.832 │ 0.07681999999999971 │      23.8 │
│ MAC001349 │ 2012-08-06 15:00:00 │     1.608 │ 0.07681999999999971 │      20.9 │
│ MAC003096 │ 2013-10-24 13:30:00 │     1.425 │               0.075 │      19.0 │
│ MAC003096 │ 2013-10-21 14:30:00 │       1.4 │               0.075 │      18.7 │
│ MAC004781 │ 2013-12-24 09:00:00 │ 2.4660001 │               0.135 │      18.3 │
│ MAC003096 │ 2013-10-21 15:00:00 │     1.344 │               0.075 │      17.9 │
│ MAC003096 │ 2013-10-22 09:30:00 │

Only 34 readings in 14 households are spikes, out of more than 155 million. They come from households that normally use very little (a 99th percentile around 0.1 kWh), and the spike values themselves are ordinary: 1 to 2.5 kWh in half an hour is what a kettle, oven or heater uses. Some repeat at similar times on nearby days, which looks like real behaviour, not a fault.

**We keep all of them.** They are real usage and add to the real bill. Removing or capping them would make those months' bills too low, and with only 34 readings they have no meaningful effect on the data as a whole.

## 10. Aggregation and integration

Our target is a monthly bill, so we don't need every half hour from here on. We add up the clean readings per household per day, join in the other sources (household details, weather, bank holidays), and save daily and monthly tables to `data/processed/`.

### 10.1 Daily usage per household

We sum the 48 half hours of each day. We also count how many of them were real readings and how many were filled, so the gaps stay visible after aggregation. Every day should have exactly 48 slots, because section 8 kept only complete months.

In [27]:
con.execute("""
    CREATE OR REPLACE TABLE daily_usage AS
    SELECT
        LCLid,
        tstp::DATE                                AS day,
        sum(kwh)                                  AS kwh,
        count(*) FILTER (WHERE status = 'real')   AS real_slots,
        count(*) FILTER (WHERE status <> 'real')  AS filled_slots
    FROM clean
    GROUP BY LCLid, day
""")

con.sql("""
    SELECT
        count(*)                                                  AS household_days,
        count(*) FILTER (WHERE real_slots + filled_slots <> 48)   AS days_without_48_slots
    FROM daily_usage
""").show()

┌────────────────┬───────────────────────┐
│ household_days │ days_without_48_slots │
│     int64      │         int64         │
├────────────────┼───────────────────────┤
│        3239805 │                     0 │
└────────────────┴───────────────────────┘



### 10.2 Household details

`informations_households.csv` gives each household's tariff (`Std` = flat rate, `ToU` = time of use) and its ACORN group, a UK classification of neighbourhoods by income and lifestyle. First we look at the values as they are.

In [28]:
con.sql(f"""
    SELECT stdorToU, Acorn_grouped, count(*) AS households
    FROM '{HOUSEHOLDS_CSV}'
    GROUP BY ALL
    ORDER BY ALL
""").show()

┌──────────┬───────────────┬────────────┐
│ stdorToU │ Acorn_grouped │ households │
│ varchar  │    varchar    │   int64    │
├──────────┼───────────────┼────────────┤
│ Std      │ ACORN-U       │         39 │
│ Std      │ Adversity     │       1518 │
│ Std      │ Affluent      │       1702 │
│ Std      │ Comfortable   │       1184 │
│ ToU      │ ACORN-        │          2 │
│ ToU      │ ACORN-U       │         10 │
│ ToU      │ Adversity     │        298 │
│ ToU      │ Affluent      │        490 │
│ ToU      │ Comfortable   │        323 │
└──────────┴───────────────┴────────────┘



Unknown ACORN groups are written two ways: `ACORN-U` and `ACORN-`. They mean the same thing, so we rename both to `Unknown`. We also give the columns clearer names and check that every household in our readings appears in this file.

In [29]:
con.execute(f"""
    CREATE OR REPLACE TABLE households AS
    SELECT
        LCLid,
        stdorToU AS tariff,
        CASE WHEN Acorn         IN ('ACORN-', 'ACORN-U') THEN 'Unknown' ELSE Acorn         END AS acorn,
        CASE WHEN Acorn_grouped IN ('ACORN-', 'ACORN-U') THEN 'Unknown' ELSE Acorn_grouped END AS acorn_group
    FROM '{HOUSEHOLDS_CSV}'
""")

con.sql("SELECT tariff, acorn_group, count(*) AS households FROM households GROUP BY ALL ORDER BY ALL").show()

# Every household in our readings should appear in the household file
con.sql("""
    SELECT count(DISTINCT LCLid) AS households_without_info
    FROM daily_usage
    WHERE LCLid NOT IN (SELECT LCLid FROM households)
""").show()

┌─────────┬─────────────┬────────────┐
│ tariff  │ acorn_group │ households │
│ varchar │   varchar   │   int64    │
├─────────┼─────────────┼────────────┤
│ Std     │ Adversity   │       1518 │
│ Std     │ Affluent    │       1702 │
│ Std     │ Comfortable │       1184 │
│ Std     │ Unknown     │         39 │
│ ToU     │ Adversity   │        298 │
│ ToU     │ Affluent    │        490 │
│ ToU     │ Comfortable │        323 │
│ ToU     │ Unknown     │         12 │
└─────────┴─────────────┴────────────┘

┌─────────────────────────┐
│ households_without_info │
│          int64          │
├─────────────────────────┤
│                       0 │
└─────────────────────────┘



### 10.3 Weather

The weather file has one row per hour, in °C. Bills depend on how cold or warm a day was, so we keep each day's average, minimum and maximum temperature. We also count the hours per day to spot missing weather data.

In [30]:
con.execute(f"""
    CREATE OR REPLACE TABLE daily_weather AS
    SELECT
        time::DATE        AS day,
        avg(temperature)  AS temp_mean,
        min(temperature)  AS temp_min,
        max(temperature)  AS temp_max,
        count(*)          AS hours
    FROM '{WEATHER_CSV}'
    GROUP BY day
""")

# Days with fewer than 24 hourly readings
con.sql("SELECT day, hours FROM daily_weather WHERE hours <> 24 ORDER BY day").show()

┌────────────┬───────┐
│    day     │ hours │
│    date    │ int64 │
├────────────┼───────┤
│ 2013-09-09 │    23 │
│ 2013-09-10 │    23 │
│ 2014-03-31 │    23 │
└────────────┴───────┘



Only three days have 23 hours instead of 24, and the last of them (2014-03-31) falls after our readings end. One missing hour barely changes a daily average, so we keep these days as they are. Clock-change days still have 24 hours, so the weather times are in UTC too, matching the meter readings.

### 10.4 Bank holidays

People use energy differently on holidays, so we flag them. First we look at the file.

In [31]:
con.sql(f"SELECT * FROM '{HOLIDAYS_CSV}' ORDER BY 1").show(max_rows=30)

┌───────────────┬──────────────────────────────────────────────┐
│ Bank holidays │                     Type                     │
│     date      │                   varchar                    │
├───────────────┼──────────────────────────────────────────────┤
│ 2012-02-01    │ New Year?s Day (substitute day)              │
│ 2012-04-06    │ Spring bank holiday (substitute day)         │
│ 2012-05-06    │ Queen?s Diamond Jubilee (extra bank holiday) │
│ 2012-06-04    │ Good Friday                                  │
│ 2012-07-05    │ Early May bank holiday                       │
│ 2012-08-27    │ Summer bank holiday                          │
│ 2012-09-04    │ Easter Monday                                │
│ 2012-12-25    │ Christmas Day                                │
│ 2012-12-26    │ Boxing Day                                   │
│ 2013-01-01    │ New Year?s Day                               │
│ 2013-01-04    │ Easter Monday                                │
│ 2013-03-29    │ Good Fr

This file has errors. Several dates have the **day and month swapped**: `2012-02-01` should be 2 January, and `2012-09-04` labelled Easter Monday should be 9 April. Some labels sit on the wrong dates too (`2012-06-04` is not Good Friday). The file also starts in 2012, so it misses the December 2011 holidays, although our readings start in November 2011.

We only need the dates, so we fix the swapped ones against the official UK bank holiday calendar, add the two missing 2011 holidays, and ignore the labels. Every UK bank holiday falls on a weekday, which gives a quick check on the result.

In [32]:
# Dates whose day and month are swapped in the file, corrected against the official UK bank holiday calendar
SWAPPED_DATES = {
    "2012-02-01": "2012-01-02",  # New Year's Day (substitute)
    "2012-05-06": "2012-06-05",  # Queen's Diamond Jubilee
    "2012-07-05": "2012-05-07",  # Early May bank holiday
    "2012-09-04": "2012-04-09",  # Easter Monday
    "2013-01-04": "2013-04-01",  # Easter Monday
    "2013-06-05": "2013-05-06",  # Early May bank holiday
}
# The file starts in 2012, but our readings start in November 2011
MISSING_DATES = ["2011-12-26", "2011-12-27"]  # Boxing Day, Christmas Day (substitute)

dates = [str(d) for (d,) in con.sql(f"SELECT \"Bank holidays\" FROM '{HOLIDAYS_CSV}'").fetchall()]
fixed = [SWAPPED_DATES.get(d, d) for d in dates] + MISSING_DATES

con.execute("CREATE OR REPLACE TABLE holidays AS SELECT unnest(?::DATE[]) AS day", [fixed])
con.sql("SELECT day, dayname(day) AS weekday FROM holidays WHERE day < '2014-03-01' ORDER BY day").show(max_rows=30)

┌────────────┬───────────┐
│    day     │  weekday  │
│    date    │  varchar  │
├────────────┼───────────┤
│ 2011-12-26 │ Monday    │
│ 2011-12-27 │ Tuesday   │
│ 2012-01-02 │ Monday    │
│ 2012-04-06 │ Friday    │
│ 2012-04-09 │ Monday    │
│ 2012-05-07 │ Monday    │
│ 2012-06-04 │ Monday    │
│ 2012-06-05 │ Tuesday   │
│ 2012-08-27 │ Monday    │
│ 2012-12-25 │ Tuesday   │
│ 2012-12-26 │ Wednesday │
│ 2013-01-01 │ Tuesday   │
│ 2013-03-29 │ Friday    │
│ 2013-04-01 │ Monday    │
│ 2013-05-06 │ Monday    │
│ 2013-05-27 │ Monday    │
│ 2013-08-26 │ Monday    │
│ 2013-12-25 │ Wednesday │
│ 2013-12-26 │ Thursday  │
│ 2014-01-01 │ Wednesday │
└────────────┴───────────┘
  20 rows      2 columns



### 10.5 Combine and save the daily table

We join everything onto the daily usage and add a **bill proxy**: daily kWh times the flat standard tariff (14.228 p/kWh). This is an estimate, not a real invoice. ToU households paid different rates in 2013, so their cost here is what they would have paid on the standard tariff. The `tariff` column lets us treat them separately later.

In [33]:
con.execute(f"""
    COPY (
        SELECT
            u.LCLid,
            u.day,
            u.kwh,
            u.kwh * {STD_TARIFF_GBP_PER_KWH}  AS cost_gbp,
            u.real_slots,
            u.filled_slots,
            h.tariff,
            h.acorn,
            h.acorn_group,
            w.temp_mean,
            w.temp_min,
            w.temp_max,
            hol.day IS NOT NULL              AS is_holiday
        FROM daily_usage AS u
        LEFT JOIN households    AS h   ON h.LCLid = u.LCLid
        LEFT JOIN daily_weather AS w   ON w.day = u.day
        LEFT JOIN holidays      AS hol ON hol.day = u.day
        ORDER BY u.LCLid, u.day
    ) TO '{DAILY_PARQUET}' (FORMAT parquet)
""")

con.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM '{DAILY_PARQUET}'")

### 10.6 Monthly table

This is the level our forecast works at: one row per household per month, with total kWh and the bill.

In [34]:
con.execute(f"""
    COPY (
        SELECT
            LCLid,
            date_trunc('month', day)::DATE  AS month,
            tariff,
            acorn,
            acorn_group,
            count(*)                        AS days,
            sum(kwh)                        AS kwh,
            sum(cost_gbp)                   AS bill_gbp,
            avg(temp_mean)                  AS temp_mean,
            sum(filled_slots)               AS filled_slots
        FROM daily
        GROUP BY ALL
        ORDER BY LCLid, month
    ) TO '{MONTHLY_PARQUET}' (FORMAT parquet)
""")

con.execute(f"CREATE OR REPLACE VIEW monthly AS SELECT * FROM '{MONTHLY_PARQUET}'")

### 10.7 Check the result

In [35]:
con.sql("""
    SELECT
        count(*)                          AS rows,
        count(DISTINCT LCLid)             AS households,
        min(day)                          AS first_day,
        max(day)                          AS last_day,
        count(*) - count(tariff)          AS missing_household_info,
        count(*) - count(temp_mean)       AS missing_weather,
        count(*) FILTER (WHERE is_holiday) AS holiday_rows
    FROM daily
""").show()

con.sql("""
    SELECT
        count(*)                       AS household_months,
        round(avg(kwh), 1)             AS avg_kwh,
        round(avg(bill_gbp), 2)        AS avg_bill_gbp,
        round(min(bill_gbp), 2)        AS min_bill_gbp,
        round(max(bill_gbp), 2)        AS max_bill_gbp
    FROM monthly
""").show()

con.sql("SELECT * FROM monthly LIMIT 5").show()

┌─────────┬────────────┬────────────┬────────────┬────────────────────────┬─────────────────┬──────────────┐
│  rows   │ households │ first_day  │  last_day  │ missing_household_info │ missing_weather │ holiday_rows │
│  int64  │   int64    │    date    │    date    │         int64          │      int64      │    int64     │
├─────────┼────────────┼────────────┼────────────┼────────────────────────┼─────────────────┼──────────────┤
│ 3239805 │       5548 │ 2011-12-01 │ 2014-01-31 │                      0 │               0 │        74917 │
└─────────┴────────────┴────────────┴────────────┴────────────────────────┴─────────────────┴──────────────┘

┌──────────────────┬─────────┬──────────────┬──────────────┬──────────────┐
│ household_months │ avg_kwh │ avg_bill_gbp │ min_bill_gbp │ max_bill_gbp │
│      int64       │ double  │    double    │    double    │    double    │
├──────────────────┼─────────┼──────────────┼──────────────┼──────────────┤
│           106225 │   311.0 │        44.

## 11. Encode categories as numbers

Most models only work with numbers, so text categories need converting. We use **one-hot encoding**: one 0/1 column per category, set to 1 when the row belongs to it. Unlike numbering the groups 1, 2, 3, this doesn't suggest an order that isn't there (Affluent is not "3 times" Adversity).

- **Tariff** has two values, so one column is enough: `tariff_tou` is 1 for time-of-use households and 0 for standard.
- **ACORN group** has four values, so it gets four columns. A linear model needs one of them left out to avoid repeating information; we keep all four here and choose which to drop when we build the model.
- The detailed `acorn` code has 18 values. That would add 18 mostly-empty columns, so we keep it as text and use the group for modelling.

We keep the original text columns too, so the table stays readable, and save the result as `monthly_encoded.parquet`.

In [36]:
con.execute(f"""
    COPY (
        SELECT
            *,
            (tariff = 'ToU')::INT              AS tariff_tou,
            (acorn_group = 'Affluent')::INT    AS acorn_affluent,
            (acorn_group = 'Comfortable')::INT AS acorn_comfortable,
            (acorn_group = 'Adversity')::INT   AS acorn_adversity,
            (acorn_group = 'Unknown')::INT     AS acorn_unknown
        FROM monthly
    ) TO '{MONTHLY_ENCODED_PARQUET}' (FORMAT parquet)
""")

con.sql(f"""
    SELECT tariff, acorn_group, tariff_tou, acorn_affluent, acorn_comfortable, acorn_adversity, acorn_unknown,
           count(*) AS household_months
    FROM '{MONTHLY_ENCODED_PARQUET}'
    GROUP BY ALL
    ORDER BY ALL
""").df()

,tariff,acorn_group,tariff_tou,acorn_affluent,acorn_comfortable,acorn_adversity,acorn_unknown,household_months
0,Std,Adversity,0,0,0,1,0,28692
1,Std,Affluent,0,1,0,0,0,33068
2,Std,Comfortable,0,0,1,0,0,22381
3,Std,Unknown,0,0,0,0,1,666
4,ToU,Adversity,1,0,0,1,0,5629
5,ToU,Affluent,1,1,0,0,0,9584
6,ToU,Comfortable,1,0,1,0,0,6004
7,ToU,Unknown,1,0,0,0,1,201


## 12. Scaling

Our numeric columns sit on very different ranges, as the table below shows: monthly kWh runs into the thousands while temperature stays within about 25 degrees. Some models (linear regression with regularisation, k-nearest neighbours, neural networks) are sensitive to that, so the features will need scaling.

We **don't scale here, on purpose**. A scaler learns the mean and spread of the data it sees. If we fit it on the whole dataset now, it would learn from the months we later use to test the forecast, which leaks future information into the model. So scaling belongs inside the Phase 3 model pipeline, fitted on the training months only and then applied to the test months.

In [37]:
con.sql("""
    SELECT
        min(kwh)       AS min_kwh,       max(kwh)       AS max_kwh,
        min(bill_gbp)  AS min_bill_gbp,  max(bill_gbp)  AS max_bill_gbp,
        min(temp_mean) AS min_temp_c,    max(temp_mean) AS max_temp_c,
        min(days)      AS min_days,      max(days)      AS max_days
    FROM monthly
""").df()

,min_kwh,max_kwh,min_bill_gbp,max_bill_gbp,min_temp_c,max_temp_c,min_days,max_days
0,0.204,6385.894001,0.029025,908.584999,3.800536,20.039099,28,31


## 13. Final checks

Each check counts rows that break a rule we set during cleaning. Every count should be 0. If any check fails, the cell stops with an error, so a broken run can't go unnoticed.

In [38]:
checks = {
    "duplicate (household, half hour) pairs":  "SELECT count(*) FROM (SELECT LCLid, tstp FROM clean GROUP BY ALL HAVING count(*) > 1)",
    "half hours off the grid":                 "SELECT count(*) FROM clean WHERE tstp <> time_bucket(INTERVAL 30 MINUTE, tstp)",
    "negative kWh":                            "SELECT count(*) FROM clean WHERE kwh < 0",
    "missing kWh":                             "SELECT count(*) FROM clean WHERE kwh IS NULL",
    "duplicate (household, day) pairs":        "SELECT count(*) FROM (SELECT LCLid, day FROM daily GROUP BY ALL HAVING count(*) > 1)",
    "days without 48 half hours":              "SELECT count(*) FROM daily WHERE real_slots + filled_slots <> 48",
    "missing values in the daily table":       "SELECT count(*) FROM daily WHERE kwh IS NULL OR tariff IS NULL OR acorn_group IS NULL OR temp_mean IS NULL",
    "incomplete months in the monthly table":  "SELECT count(*) FROM monthly WHERE days <> day(last_day(month))",
}

for name, sql in checks.items():
    problems = con.sql(sql).fetchone()[0]
    print(f"{'OK  ' if problems == 0 else 'FAIL'}  {name}: {problems}")

    assert problems == 0, f"Check failed: {name}"

OK    duplicate (household, half hour) pairs: 0
OK    half hours off the grid: 0
OK    negative kWh: 0
OK    missing kWh: 0
OK    duplicate (household, day) pairs: 0
OK    days without 48 half hours: 0
OK    missing values in the daily table: 0
OK    incomplete months in the monthly table: 0
